<a href="https://colab.research.google.com/github/MayerT1/PIPECAST/blob/flood_mapping/flood_mapping/notebooks/Alaska_PIPECAST_january.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction

In this module, we will take a sample image from ICEYE and run produce a binary water map. We will do this by following these steps

1. Obtain ICEYE image
2. Speckle Filter the image
3. Psuedo Terrain correct the image
4. Determine initial threshold through sampling points that have recurring water using JRC
5. Run HYDRAFloods Edge Otsu algorithm

# Part 1: Housekeeping and obtaining ICEYE image

In [ ]:
my_gee_folder = 'users/mickymags/watchduty/'

In [ ]:
!pip install ipyleaflet==0.18.2 geemap hydrafloods     # Install hydrafloods and its relevant dependencies
!pip install geemap

In [ ]:
from hydrafloods import corrections
import hydrafloods as hf
import ee
import geemap
import matplotlib.pyplot as plt

In [ ]:
ee.Authenticate()
ee.Initialize(project = 'servir-sco-assets')

In [ ]:
aoi = ee.FeatureCollection('users/mickymags/watchduty/ak_aoi1')

aoi_geom = aoi.geometry()

# Get the coordinates of the center of the AOI for mapping purposes
aoi_centroid = aoi.geometry().centroid()             # Get the center of the AOI
lon = aoi_centroid.coordinates().get(0).getInfo()    # Extract the longitude from the centroid
lat = aoi_centroid.coordinates().get(1).getInfo()    # Extract the latitude from the centroid

In [ ]:
iceye = hf.Dataset(
    region = aoi_geom,               #my_geom
    start_time = '2025-10-13',
    end_time = '2025-10-15',
    asset_id = 'users/mickymags/watchduty/alaska_iceye_v2'
)

In [ ]:
iceye_med = iceye.collection.median()

In [ ]:
iceye_med = iceye_med.rename('VV', 'angle').clip(aoi_geom)

In [ ]:
iceye_med.bandNames().getInfo()

In [ ]:
iceye_vp = {
    'bands': 'VV', #'b1',
    'min': 0,
    'max': 2000
}

In [ ]:
ic_lee_med = hf.lee_sigma(iceye_med)
ic_gamma_med = hf.gamma_map(iceye_med)
ic_refined_med = hf.refined_lee(iceye_med, keep_bands=['angle'])
ic_pmedian_med = hf.p_median(iceye_med)

In [ ]:
ic_lee_med.bandNames().getInfo()

# Histogram Analysis

In [ ]:
# Load your SAR image from assets
asset_id1 = "users/mickymags/watchduty/alaska_iceye_v2/kipnuk_1014_0112"

image = ee.Image(asset_id1)
image = image.rename('VV', 'angle')

# Get band names
bands = image.select('VV').bandNames().getInfo()
print("Bands in image:", bands)

In [ ]:
# Create a histogram for each band
def plot_histograms(image, bands,  aoi_descriptor, region, xmin, xmax, scale=3.46325, n_bins=4096):
    """Plot histograms for each band of an ee.Image."""
    fig, axes = plt.subplots(len(bands), 1, figsize=(8, 4*len(bands)))

    if len(bands) == 1:
        axes = [axes]

    for i, band in enumerate(bands):
        # Reduce region to histogram
        hist_dict = image.select(band).reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=n_bins), #maxBuckets=n_bins),
            geometry=region,
            scale=scale,

            bestEffort=True
        ).get(band).getInfo()

        if hist_dict is not None:
            counts = hist_dict['histogram']
            bucketMeans = hist_dict['bucketMeans']

            axes[i].bar(bucketMeans, counts, width=(bucketMeans[1]-bucketMeans[0]))
            axes[i].set_title(f"Histogram of {band} for {aoi_descriptor}")
            axes[i].set_xlabel("Pixel values")
            axes[i].set_ylabel("Frequency")
            axes[i].set_xlim(xmin, xmax) #0, 250
        else:
            axes[i].set_title(f"{band}: No data")

    plt.grid()
    plt.tight_layout()
    plt.show()

In [ ]:
sent1 = ee.ImageCollection("COPERNICUS/S1_GRD").filterBounds(aoi_geom).filterDate("2025-10-22", "2025-10-23").select(["VV", "angle"]).first().clip(aoi_geom)
s1_rl = hf.refined_lee(sent1)

In [ ]:
plot_histograms(s1_rl, bands, 'Sentinel-1 Image', aoi_geom, -30, 0, scale = 30, n_bins = 256)

In [ ]:
plot_histograms(image, bands, 'Raw ICEYE Image 1', aoi_geom, 0, 3000, scale = 4)

In [ ]:
plot_histograms(ic_lee_med, bands, 'ICEYE Image 1 (10/12)', aoi_geom, 0, 3000, scale = 4, n_bins = 256)

In [ ]:
plot_histograms(ic_refined_med, bands, 'Refined-Lee Image', aoi_geom, 250, 400, scale = 4, n_bins = 48)

In [ ]:
plot_histograms(ic_gamma_med, bands, 'Gamma-Map Filtered Image', aoi_geom, 0, 3000, scale = 4, n_bins = 48)

In [ ]:
plot_histograms(ic_pmedian_med, bands, 'ICEYE PMedian', aoi_geom, 0, 3000, scale = 4, n_bins = 2048)

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 11)
Map.addLayer(aoi_geom, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE2')
Map.addLayer(ic_pmedian_med, iceye_vp, 'ICEYE2 P Median')
Map.addLayer(ic_lee_med, iceye_vp, 'ICEYE Lee')
Map.addLayer(ic_gamma_med, iceye_vp, 'ICEYE Gamma')
#Map.addLayer(s1_rl, iceye_vp, 'Sentinel-1 Refined Lee')
#Map.addLayer(iceye2_lee, iceye_vp, 'ICEYE2 Lee')
#Map.addLayer(iceye2_gamma, iceye_vp, 'ICEYE2 Gamma')
#Map.addLayer(iceye2_refined, iceye_vp, 'ICEYE2 Refined')

Map.addLayerControl()
Map

# Part 4: Initial Threshold determination

### Raw Image

In [ ]:
import numpy as np

In [ ]:
iceye_sampled = iceye_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  vals.append(feat_of_int['properties']['VV'])

np_vals = np.array(vals)
np_vals.mean()

In [ ]:
edge = hf.edge_otsu(
    iceye_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_vals.mean(),
    thresh_no_data = np_vals.mean()+50,
    scale = 4
)

### P-Median Image

In [ ]:
iceye_sampled = ic_pmedian_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

pmed_vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  pmed_vals.append(feat_of_int['properties']['VV'])

np_pmed = np.array(pmed_vals)
np_pmed.mean()

In [ ]:
edge_median = hf.edge_otsu(
    ic_pmedian_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_pmed.mean(),
    thresh_no_data = np_pmed.mean()+50,
)

# Lee-Sigma Image

In [ ]:
iceye_sampled = ic_lee_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
features = iceye_sampled['features']
first_feature = features[0]
first_feature['properties']['VV']

lee_vals = []

for j in range(len(features)):
  feat_of_int = features[j]
  lee_vals.append(feat_of_int['properties']['VV'])

np_lee = np.array(lee_vals)
np_lee.mean()

In [ ]:
edge_lee = hf.edge_otsu(
    ic_lee_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_lee.mean(),
    thresh_no_data = np_lee.mean()+50,
)

# Refined-Lee Image

In [ ]:
iceye_refined_sampled = ic_refined_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
refined_features = iceye_refined_sampled['features']
first_re_feature = refined_features[0]
first_re_feature['properties']['VV']

refined_vals = []

for j in range(len(refined_features)):
  refined_feat_of_int = refined_features[j]
  refined_vals.append(refined_feat_of_int['properties']['VV'])

np_refined = np.array(refined_vals)
np_refined.mean()

In [ ]:
edge_refined = hf.edge_otsu(
    ic_refined_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_refined.mean(),
    thresh_no_data = np_refined.mean()+50,
)

### Gamma MAP Image

In [ ]:
iceye_gamma_sampled = ic_gamma_med.sample(aoi, numPixels = 5000, scale = 4).getInfo()
gamma_features = iceye_gamma_sampled['features']
first_gamma_feature = refined_features[0]
first_gamma_feature['properties']['VV']

gamma_vals = []

for j in range(len(gamma_features)):
  gamma_feat_of_int = gamma_features[j]
  gamma_vals.append(gamma_feat_of_int['properties']['VV'])

np_gamma = np.array(gamma_vals)
np_gamma.mean()

In [ ]:
edge_gamma = hf.edge_otsu(
    ic_gamma_med, #iceye_med
    band = 'VV',
    region = aoi,
    edge_buffer = 30,
    initial_threshold = np_gamma.mean(),
    thresh_no_data = np_gamma.mean()+50,
)

### Map Visualization

In [ ]:
water_vp = {
    'palette': ['000000', 'add8e6'],
    'min': 0,
    'max': 1
}

In [ ]:
Map = geemap.Map(center = (lat, lon), zoom = 12)
Map.addLayer(aoi, {}, 'Area of Interest')
Map.addLayer(iceye_med, iceye_vp, 'ICEYE')
Map.addLayer(edge, water_vp, 'Edge')
Map.addLayer(edge_median, water_vp, 'Edge Median')
Map.addLayer(edge_lee, water_vp, 'Edge Lee')
Map.addLayer(edge_refined, water_vp, 'Edge Refined')
Map.addLayerControl()
Map

# Exporting Images to GEE Assets

In [ ]:
geemap.ee_export_image_to_asset(image =edge,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_no_sf',
                                description = 'iceye_edge_otsu_no_speckle_filter',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_lee,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_leesigma_sf',
                                description = 'edge_no_tc_leesigma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_median,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_median_sf',
                                description = 'iedge_no_tc_median_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_refined,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_refined_sf',
                                description = 'edge_no_tc_refined_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

In [ ]:
geemap.ee_export_image_to_asset(image = edge_gamma,
                                #region = export_aoi,
                                assetId = 'users/mickymags/watchduty/edge_no_tc_gamma_sf',
                                description = 'edge_no_tc_gamma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels = 1e13)

# Exporting Images to Google Drive

In [ ]:
geemap.ee_export_image_to_drive(image=edge_gamma,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_gamma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_no_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_lee,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_leesigma_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_refined,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_refined_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)

In [ ]:
geemap.ee_export_image_to_drive(image=edge_median,
                                folder='WatchDuty',
                                description='gdrive_edge_no_tc_median_sf',
                                scale=4,
                                crs='EPSG:32603',
                                maxPixels=1e13)